# MethylGPT: Age Prediction Finetuning

This notebook demonstrates how to finetune MethylGPT for **biological age prediction** on the [AltumAge](https://github.com/rsinghlab/AltumAge) dataset using PyTorch Lightning.

MethylGPT learns rich representations of DNA methylation profiles through masked-value pretraining. By attaching a lightweight regression head and finetuning on age-labeled samples, we can predict chronological age from methylation arrays.

**Outline:**
1. Setup paths & download data
2. Define dataset & model classes
3. Load and prepare AltumAge data
4. Build model with age prediction head
5. Training loop (3-epoch demo)
6. Evaluate on test set

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q methylgpt[tutorials]
    !pip install -q gdown
    import torch
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("WARNING: No GPU. Go to Runtime > Change runtime type > GPU")

In [ ]:
import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn import preprocessing

from methylgpt import MethylGPTModel, MethylVocab
from scgpt.tokenizer import tokenize_and_pad_batch, random_mask_value

warnings.filterwarnings("ignore", message=".*IProgress.*")
warnings.filterwarnings("ignore", message=".*flash_attn.*")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Setup Paths & Download Data

Before running this notebook, execute the `download_files.sh` script in the tutorial directory to download:
- The pretrained MethylGPT model weights (`pretrained_models/tiny/`)
- The AltumAge dataset splits (`data/altumage_metadata/`)
- The probe ID file (`probe_ids_type3.csv`)

```bash
cd tutorials/finetuning_age_prediction
bash download_files.sh
```

In [ ]:
# === UPDATE THESE PATHS ===
SCRIPT_DIR = Path(".")  # Tutorial directory
MODEL_DIR = SCRIPT_DIR / "pretrained_models" / "tiny"
DATA_DIR = SCRIPT_DIR / "data" / "altumage_metadata"
PROBE_ID_FILE = SCRIPT_DIR / "probe_ids_type3.csv"

# Verify files exist
for p in [MODEL_DIR / "args.json", PROBE_ID_FILE]:
    assert p.exists(), f"File not found: {p}. Run download_files.sh first."

# Load model config
with open(MODEL_DIR / "args.json", "r") as f:
    pretrain_config = json.load(f)

model_file = list(MODEL_DIR.glob("*.pt"))
assert model_file, f"No .pt files in {MODEL_DIR}"
model_file = str(model_file[0])
print(f"Model: {model_file}")
print(f"Data: {DATA_DIR}")

## 2. Define Dataset & Model

In [ ]:
class AgeDataset(torch.utils.data.Dataset):
    """Dataset for age prediction finetuning."""
    
    def __init__(self, vocab_obj, df, scaler, mask_ratio=0.15, mask_seed=42):
        self.vocab = vocab_obj
        self.scaler = scaler
        self.mask_ratio = mask_ratio
        self.mask_seed = mask_seed
        self.gene_datas = df["data"].to_list()
        self.ages = df["age"].to_numpy()
        self.ages_norm = torch.tensor(
            scaler.transform(self.ages.reshape(-1, 1)), dtype=torch.float
        )
    
    def __getitem__(self, index):
        return self.gene_datas[index], torch.tensor(self.ages[index]).float(), self.ages_norm[index]
    
    def __len__(self):
        return len(self.ages)
    
    def collater(self, batch):
        gene_datas, ages, ages_norm = zip(*batch)
        gene_ids, masked_values, target_values = self._tokenize(torch.tensor(gene_datas))
        return gene_ids, masked_values, target_values, torch.stack(ages), torch.stack(ages_norm)
    
    def _tokenize(self, data):
        data = torch.nan_to_num(data, nan=-2.0)
        if isinstance(data, torch.Tensor):
            data = data.numpy()
        tokenized = tokenize_and_pad_batch(
            data, self.vocab.CpG_ids,
            max_len=self.vocab.max_seq_len,
            vocab=self.vocab.vocab,
            pad_token="<pad>", pad_value=-2,
            append_cls=True, include_zero_gene=True,
        )
        masked = random_mask_value(
            tokenized["values"], mask_ratio=self.mask_ratio,
            mask_value=-1, pad_value=-2, mask_seed=self.mask_seed,
        )
        return tokenized["genes"], masked, tokenized["values"]


class VocabWrapper:
    """Lightweight vocab wrapper for finetuning datasets."""
    
    def __init__(self, probe_id_file, n_hvg):
        from torchtext.vocab import Vocab
        from torchtext._torchtext import Vocab as VocabPybind
        
        special_tokens = ["<pad>", "<cls>", "<eoc>"]
        CpG_list = pd.read_csv(probe_id_file)["illumina_probe_id"].values.tolist()
        self.CpG_ids = len(special_tokens) + np.arange(len(CpG_list))
        self.vocab = Vocab(VocabPybind(special_tokens + CpG_list, None))
        self.vocab.set_default_index(self.vocab["<pad>"])
        self.max_seq_len = n_hvg + 1
        self.pad_token = "<pad>"
        self.pad_value = -2

print("Dataset and model classes defined.")

## 3. Load Data

In [ ]:
# Load data splits
train_df = pd.read_parquet(DATA_DIR / "altumage_train.parquet")
valid_df = pd.read_parquet(DATA_DIR / "altumage_valid.parquet")
test_df = pd.read_parquet(DATA_DIR / "altumage_test.parquet")

print(f"Train: {len(train_df)} samples, Valid: {len(valid_df)}, Test: {len(test_df)}")
print(f"Age range: {train_df['age'].min():.0f} - {train_df['age'].max():.0f} years")

# Fit scaler on training data
scaler = preprocessing.MinMaxScaler(feature_range=(0, 1))
scaler.fit(train_df["age"].to_numpy().reshape(-1, 1))

# Create vocab wrapper
vocab_wrapper = VocabWrapper(str(PROBE_ID_FILE), pretrain_config["n_hvg"])

# Create datasets
train_dataset = AgeDataset(vocab_wrapper, train_df, scaler, mask_ratio=0.15)
valid_dataset = AgeDataset(vocab_wrapper, valid_df, scaler, mask_ratio=0.0)
test_dataset = AgeDataset(vocab_wrapper, test_df, scaler, mask_ratio=0.0)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=2, collate_fn=train_dataset.collater, shuffle=True, drop_last=True, num_workers=0)
valid_loader = DataLoader(valid_dataset, batch_size=2, collate_fn=valid_dataset.collater, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=2, collate_fn=test_dataset.collater, num_workers=0)

print("Data loaded and ready.")

## 4. Build Model

In [ ]:
from scgpt.model.model import TransformerModel

# Load pretrained backbone
backbone = TransformerModel(
    len(vocab_wrapper.vocab), pretrain_config["layer_size"],
    pretrain_config["nhead"], pretrain_config["layer_size"],
    pretrain_config["nlayers"], vocab=vocab_wrapper.vocab,
    dropout=pretrain_config.get("dropout", 0.0), pad_token="<pad>",
    pad_value=-2, do_mvc=True, do_dab=False,
    use_batch_labels=False, num_batch_labels=None,
    domain_spec_batchnorm=False, n_input_bins=None,
    ecs_threshold=0.0, explicit_zero_prob=False,
    use_fast_transformer=pretrain_config.get("fast_transformer", False),
    pre_norm=pretrain_config.get("pre_norm", False),
)

# Load pretrained weights
try:
    backbone.load_state_dict(torch.load(model_file, map_location="cpu"))
    print(f"Loaded all params from {model_file}")
except RuntimeError:
    model_dict = backbone.state_dict()
    pretrained_dict = torch.load(model_file, map_location="cpu")
    compatible = {k: v for k, v in pretrained_dict.items() if k in model_dict and v.shape == model_dict[k].shape}
    model_dict.update(compatible)
    backbone.load_state_dict(model_dict)
    print(f"Loaded {len(compatible)}/{len(pretrained_dict)} compatible params")

# Simple age prediction head
class AgePredictor(nn.Module):
    def __init__(self, backbone, embed_dim):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 1),
            nn.Sigmoid(),
        )
    
    def forward(self, gene_ids, values):
        src_key_padding_mask = gene_ids.eq(vocab_wrapper.vocab["<pad>"])
        output = self.backbone(gene_ids, values, src_key_padding_mask=src_key_padding_mask, batch_labels=None)
        cell_emb = output["cell_emb"]
        return self.head(cell_emb).squeeze(-1)

age_model = AgePredictor(backbone, pretrain_config["layer_size"]).to(device)
print(f"Model ready: {sum(p.numel() for p in age_model.parameters()):,} parameters")

## 5. Training Loop

In [ ]:
# Optimizer with differential LR
optimizer = torch.optim.Adam([
    {"params": age_model.backbone.parameters(), "lr": 1e-5},
    {"params": age_model.head.parameters(), "lr": 1e-4},
], weight_decay=0.01)

criterion = nn.MSELoss()
NUM_EPOCHS = 3  # Demo: use 50+ for real training

# Use mixed precision for flash attention compatibility and memory efficiency
scaler_amp = torch.cuda.amp.GradScaler()

for epoch in range(NUM_EPOCHS):
    # Train
    age_model.train()
    train_losses = []
    for batch_idx, (gene_ids, masked_vals, target_vals, ages, ages_norm) in enumerate(train_loader):
        gene_ids = gene_ids.to(device)
        masked_vals = masked_vals.to(device)
        ages_norm = ages_norm.to(device).squeeze()
        
        with torch.cuda.amp.autocast():
            pred = age_model(gene_ids, masked_vals)
            loss = criterion(pred, ages_norm)
        
        optimizer.zero_grad()
        scaler_amp.scale(loss).backward()
        scaler_amp.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(age_model.parameters(), 1.0)
        scaler_amp.step(optimizer)
        scaler_amp.update()
        
        train_losses.append(loss.item())
        if (batch_idx + 1) % 10 == 0:
            print(f"  Epoch {epoch+1}, Batch {batch_idx+1}, Loss: {np.mean(train_losses[-10:]):.4f}")
    
    # Validate
    age_model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for gene_ids, masked_vals, target_vals, ages, ages_norm in valid_loader:
            gene_ids = gene_ids.to(device)
            target_vals = target_vals.to(device)
            with torch.cuda.amp.autocast():
                pred = age_model(gene_ids, target_vals)
            # Inverse transform
            pred_age = scaler.inverse_transform(pred.float().cpu().numpy().reshape(-1, 1)).flatten()
            all_preds.extend(pred_age)
            all_labels.extend(ages.numpy())
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    mae = np.mean(np.abs(all_preds - all_labels))
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} — Train loss: {np.mean(train_losses):.4f}, Valid MAE: {mae:.2f} years")

## 6. Evaluate on Test Set

In [ ]:
# Evaluate on test set
age_model.eval()
test_preds, test_labels = [], []
with torch.no_grad():
    for gene_ids, masked_vals, target_vals, ages, ages_norm in test_loader:
        gene_ids = gene_ids.to(device)
        target_vals = target_vals.to(device)
        with torch.cuda.amp.autocast():
            pred = age_model(gene_ids, target_vals)
        pred_age = scaler.inverse_transform(pred.float().cpu().numpy().reshape(-1, 1)).flatten()
        test_preds.extend(pred_age)
        test_labels.extend(ages.numpy())

test_preds = np.array(test_preds)
test_labels = np.array(test_labels)

# Metrics
from sklearn.metrics import r2_score, mean_absolute_error
from scipy.stats import pearsonr

mae = mean_absolute_error(test_labels, test_preds)
r2 = r2_score(test_labels, test_preds)
r, _ = pearsonr(test_labels, test_preds)

print(f"Test MAE: {mae:.2f} years")
print(f"Test R²: {r2:.3f}")
print(f"Test Pearson r: {r:.3f}")

# Scatter plot with aquarel theme
import matplotlib.pyplot as plt
from aquarel import load_theme

theme = (
    load_theme("scientific")
    .set_grid(draw=False)
    .set_font(size=15)
    .set_ticks(direction="out")
    .set_axis_labels(pad=10)
)
theme.apply()

fig, ax = plt.subplots(figsize=(6, 6))
# Outline layer
ax.scatter(test_labels, test_preds, s=15, c="black", alpha=1, zorder=1)
# Data layer
ax.scatter(test_labels, test_preds, s=10, alpha=0.5, c="steelblue", zorder=2)
lims = [min(test_labels.min(), test_preds.min()), max(test_labels.max(), test_preds.max())]
ax.plot(lims, lims, "r--", alpha=0.8, label="y = x")
ax.set_xlabel("Chronological Age (years)")
ax.set_ylabel("Predicted Age (years)")
ax.set_title(f"Age Prediction (MAE={mae:.1f}, r={r:.2f})")
ax.legend(frameon=False)

theme.apply_transforms()

plt.savefig("age_prediction_scatter.pdf", bbox_inches="tight")
plt.savefig("age_prediction_scatter.png", dpi=600, bbox_inches="tight")
plt.show()

## Next Steps

This demo used only 3 epochs with a batch size of 2. For production-quality age prediction:

- **Full training**: Use `run_finetuning_altumage.sh` which trains for 50+ epochs with larger batch sizes
- **Hyperparameters**: See `train_methyGPT_altumage.yml` for the full configuration
- **PyTorch Lightning**: The full pipeline in `finetuning_age_main.py` uses Lightning for distributed training, checkpointing, and logging

**Other tutorials:**
- `tutorials/pretraining/` -- Pretrain MethylGPT from scratch
- `tutorials/embeddings/` -- Extract methylation embeddings
- `tutorials/imputation/` -- Impute missing CpG values